In [ ]:
# ==========================================
# CREADOR DE ARCHIVOS DE PRUEBA
# ==========================================
# Generamos los archivos requeridos según las especificaciones del taller

with open("estadoinicial.txt", "w") as f:
    f.write("326\n504\n187")

with open("estadofinal.txt", "w") as f:
    f.write("847\n265\n310")

print("Archivos de prueba creados exitosamente.")

Archivos de prueba creados exitosamente.


In [ ]:
import heapq

# ==========================================
# FUNCIONES AUXILIARES DE LECTURA Y FORMATO
# ==========================================

def cargar_estado(filepath):
    """Lee un archivo .txt y lo convierte en una tupla de tuplas 3x3"""
    with open(filepath, 'r') as f:
        lineas = [linea.strip() for linea in f.readlines() if linea.strip()]
    matriz = []
    for linea in lineas:
        matriz.append(tuple(int(c) for c in linea))
    return tuple(matriz)

def imprimir_tablero(estado):
    """Muestra el estado en formato de matriz 3x3"""
    for fila in estado:
        print(" ".join(str(elem) if elem != 0 else "_" for elem in fila))
    print()

def obtener_posicion(estado, valor):
    """Encuentra la posición (fila, col) de un número en la matriz"""
    for r in range(3):
        for c in range(3):
            if estado[r][c] == valor:
                return r, c
    return None

# ==========================================
# HEURÍSTICA Y GENERACIÓN DE HIJOS
# ==========================================

def heuristica_manhattan_conflictos(estado, estado_final):
    """
    Calcula la distancia Manhattan + Penalización por conflictos lineales.
    """
    distancia = 0
    # 1. Distancia Manhattan
    for r in range(3):
        for c in range(3):
            val = estado[r][c]
            if val != 0:
                target_r, target_c = obtener_posicion(estado_final, val)
                distancia += abs(r - target_r) + abs(c - target_c)
                
    # 2. Conflictos Lineales (Filas)
    for r in range(3):
        for c1 in range(3):
            for c2 in range(c1 + 1, 3):
                val1, val2 = estado[r][c1], estado[r][c2]
                if val1 != 0 and val2 != 0:
                    tr1, tc1 = obtener_posicion(estado_final, val1)
                    tr2, tc2 = obtener_posicion(estado_final, val2)
                    if tr1 == r and tr2 == r and tc1 > tc2:
                        distancia += 2
                        
    return distancia

def obtener_sucesores(estado):
    """Genera los estados vecinos desplazando el espacio vacío (0)"""
    r_vacio, c_vacio = obtener_posicion(estado, 0)
    movimientos = [(-1, 0, 'Abajo'), (1, 0, 'Arriba'), (0, -1, 'Derecha'), (0, 1, 'Izquierda')]
    sucesores = []
    
    for dr, dc, direccion in movimientos:
        nr, nc = r_vacio + dr, c_vacio + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            # Crear nueva matriz clonada con el movimiento
            nueva_matriz = [list(fila) for fila in estado]
            # Intercambiar ficha con el espacio vacío
            nueva_matriz[r_vacio][c_vacio], nueva_matriz[nr][nc] = nueva_matriz[nr][nc], nueva_matriz[r_vacio][c_vacio]
            
            # Formato inmutable (tupla de tuplas)
            nuevo_estado = tuple(tuple(fila) for fila in nueva_matriz)
            ficha_movida = estado[nr][nc]
            sucesores.append((nuevo_estado, f"Mover ficha {ficha_movida} hacia {direccion}"))
            
    return sucesores

# ==========================================
# ALGORITMO BÚSQUEDA A*
# ==========================================

def resolver_8_puzzle(archivo_inicial, archivo_final):
    estado_inicial = cargar_estado(archivo_inicial)
    estado_final = cargar_estado(archivo_final)
    
    # Priority Queue: (f_score, contador_id, estado_actual, g_score, camino)
    count = 0
    open_set = []
    h_inicial = heuristica_manhattan_conflictos(estado_inicial, estado_final)
    
    # Insertar el nodo inicial
    heapq.heappush(open_set, (h_inicial, count, estado_inicial, 0, []))
    
    costos_g = {estado_inicial: 0}
    visitados = set()
    iteraciones = 0
    
    while open_set:
        iteraciones += 1
        _, _, actual, g, camino = heapq.heappop(open_set)
        
        if actual in visitados:
            continue
        visitados.add(actual)
        
        # ¿Llegamos al estado final?
        if actual == estado_final:
            return camino, iteraciones, actual
        
        for vecino, paso in obtener_sucesores(actual):
            nuevo_g = g + 1
            
            if vecino not in costos_g or nuevo_g < costos_g[vecino]:
                costos_g[vecino] = nuevo_g
                h = heuristica_manhattan_conflictos(vecino, estado_final)
                f = nuevo_g + h
                count += 1
                heapq.heappush(open_set, (f, count, vecino, nuevo_g, camino + [(paso, vecino)]))
                
    return None, iteraciones, None

# ==========================================
# EJECUCIÓN Y MOSTRAR RESULTADOS
# ==========================================

camino_solucion, total_iteraciones, final = resolver_8_puzzle("estadoinicial.txt", "estadofinal.txt")

print("==========================================")
print("     RESULTADOS DE LA BÚSQUEDA A*        ")
print("==========================================\n")
print(f"Número total de iteraciones realizadas: {total_iteraciones}\n")

if camino_solucion is not None:
    print(f"Solución encontrada en {len(camino_solucion)} pasos:\n")
    print("--- ESTADO INICIAL ---")
    imprimir_tablero(cargar_estado("estadoinicial.txt"))
    
    for i, (movimiento, estado) in enumerate(camino_solucion, 1):
        print(f"Paso {i}: {movimiento}")
        imprimir_tablero(estado)
else:
    print("No se encontró ninguna solución para el tablero dado.")

     RESULTADOS DE LA BÚSQUEDA A*        

Número total de iteraciones realizadas: 309

Solución encontrada en 20 pasos:

--- ESTADO INICIAL ---
3 2 6
5 _ 4
1 8 7

Paso 1: Mover ficha 8 hacia Abajo
3 2 6
5 8 4
1 _ 7

Paso 2: Mover ficha 1 hacia Izquierda
3 2 6
5 8 4
_ 1 7

Paso 3: Mover ficha 5 hacia Arriba
3 2 6
_ 8 4
5 1 7

Paso 4: Mover ficha 3 hacia Arriba
_ 2 6
3 8 4
5 1 7

Paso 5: Mover ficha 2 hacia Derecha
2 _ 6
3 8 4
5 1 7

Paso 6: Mover ficha 8 hacia Abajo
2 8 6
3 _ 4
5 1 7

Paso 7: Mover ficha 1 hacia Abajo
2 8 6
3 1 4
5 _ 7

Paso 8: Mover ficha 5 hacia Izquierda
2 8 6
3 1 4
_ 5 7

Paso 9: Mover ficha 3 hacia Arriba
2 8 6
_ 1 4
3 5 7

Paso 10: Mover ficha 2 hacia Arriba
_ 8 6
2 1 4
3 5 7

Paso 11: Mover ficha 8 hacia Derecha
8 _ 6
2 1 4
3 5 7

Paso 12: Mover ficha 6 hacia Derecha
8 6 _
2 1 4
3 5 7

Paso 13: Mover ficha 4 hacia Abajo
8 6 4
2 1 _
3 5 7

Paso 14: Mover ficha 7 hacia Abajo
8 6 4
2 1 7
3 5 _

Paso 15: Mover ficha 5 hacia Izquierda
8 6 4
2 1 7
3 _ 5

Paso 16: Move